In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# AGENT ROLE MAPPINGS & SYNERGIES (PHASE 2 & 3)
# ============================================================================

AGENT_ROLES = {
    'Jett': 'Duelist', 'Raze': 'Duelist', 'Phoenix': 'Duelist', 
    'Reyna': 'Duelist', 'Neon': 'Duelist', 'Yoru': 'Duelist', 'Iso': 'Duelist',
    'Omen': 'Controller', 'Brimstone': 'Controller', 'Viper': 'Controller', 
    'Astra': 'Controller', 'Harbor': 'Controller', 'Vyse': 'Controller',
    'Sova': 'Initiator', 'Breach': 'Initiator', 'KAY/O': 'Initiator', 
    'Fade': 'Initiator', 'Skye': 'Initiator', 'Gekko': 'Initiator',
    'Sage': 'Sentinel', 'Killjoy': 'Sentinel', 'Cypher': 'Sentinel', 
    'Chamber': 'Sentinel', 'Clove': 'Sentinel', 'Deadlock': 'Sentinel'
}

# Common synergies: agent duos that work well together
AGENT_SYNERGIES = [
    ('Jett', 'Sova'),
    ('Raze', 'Fade'),
    ('Viper', 'Cypher'),
    ('Omen', 'Sage'),
    ('Brimstone', 'Killjoy'),
    ('Chamber', 'Harbor'),
    ('Astra', 'Breach'),
    ('Reyna', 'Skye'),
    ('KAY/O', 'Phoenix'),
    ('Neon', 'Harbor'),
]

# ============================================================================
# PHASE 1: DATA LOADING & CLEANING
# ============================================================================

matches_df = pd.read_csv('data/processed/matches.csv')
maps_df = pd.read_csv('data/processed/maps.csv')
player_df = pd.read_csv('data/processed/player_stats.csv')

print("=" * 70)
print("PHASE 1: DATA LOADING & CLEANING")
print("=" * 70)
print("MATCHES columns:", matches_df.columns.tolist())
print("MAPS columns:", maps_df.columns.tolist())
print("PLAYERS columns:", player_df.columns.tolist())

# Data cleaning
if 'match_id' in matches_df.columns:
    matches_df['match_id'] = matches_df['match_id'].astype(str).str.extract(r'(\d+)')[0]

if 'match_id' in maps_df.columns:
    maps_df['match_id'] = maps_df['match_id'].astype(str).str.extract(r'(\d+)')[0]

if 'map_id' in maps_df.columns:
    maps_df['map_id'] = maps_df['map_id'].astype(str).str.strip()

if 'map_id' in player_df.columns:
    player_df['map_id'] = player_df['map_id'].astype(str).str.strip()

# Map name cleaning
maps_df['map_name'] = maps_df['map_name'].astype(str).str.lower().str.strip()
valid_maps = ['ascent', 'bind', 'haven', 'split', 'lotus', 'sunset', 'abyss', 'icebox', 'fracture', 'breeze', 'pearl']
maps_df['map_name'] = maps_df['map_name'].apply(lambda x: next((m for m in valid_maps if m in x), np.nan))
maps_df = maps_df.dropna(subset=['map_name'])

# Merge: Matches -> Maps
df_train = pd.merge(matches_df, maps_df, on='match_id', how='inner')

# Define target
df_train['team_a_won_map'] = (df_train['team_a_score'] > df_train['team_b_score']).astype(int)

# Merge: df_train -> Players
player_df['agent'] = player_df['agent'].astype(str).str.split(',').str[0].str.strip()
team_agents = pd.get_dummies(player_df[['map_id', 'team', 'agent']], 
                             columns=['agent'], prefix='', prefix_sep='').groupby(['map_id', 'team']).max().reset_index()

team_a = team_agents[team_agents['team'] == 'team_a'].drop('team', axis=1).rename(columns=lambda x: x if x == 'map_id' else f"{x}_t1")
team_b = team_agents[team_agents['team'] == 'team_b'].drop('team', axis=1).rename(columns=lambda x: x if x == 'map_id' else f"{x}_t2")

df_train = df_train.merge(team_a, on='map_id', how='left').merge(team_b, on='map_id', how='left').fillna(0)

print(f"✓ Training data rows: {len(df_train)}")
assert len(df_train) > 0, "Training data is empty!"

# ============================================================================
# PHASE 1: MAP ONE-HOT ENCODING & DIFFERENTIAL FEATURES
# ============================================================================

print("\n" + "=" * 70)
print("PHASE 1: DIFFERENTIAL FEATURES & MODEL ARCHITECTURE")
print("=" * 70)

# Map encoding
map_dummies = pd.get_dummies(df_train['map_name'], prefix='map')
df_train = pd.concat([df_train, map_dummies], axis=1)

# Separate column categories
map_cols = list(map_dummies.columns)
t1_agent_cols = [col for col in df_train.columns if col.endswith('_t1') and col != 'team_t1']
t2_agent_cols = [col for col in df_train.columns if col.endswith('_t2') and col != 'team_t2']

# Create differential features for individual agents
diff_cols = []
for col in t1_agent_cols:
    agent = col.replace('_t1', '')
    t2_col = f"{agent}_t2"
    diff_col = f"{agent}_diff"
    
    if t2_col in df_train.columns:
        df_train[diff_col] = df_train[col] - df_train[t2_col]
        diff_cols.append(diff_col)

print(f"✓ Created {len(diff_cols)} individual agent differential features")

# ============================================================================
# PHASE 2: ROLE-BASED DIFFERENTIAL FEATURES
# ============================================================================

print("\n" + "=" * 70)
print("PHASE 2: ROLE-BASED FEATURES")
print("=" * 70)

role_features = {}

# For each role, calculate team composition differences
for role in ['Duelist', 'Controller', 'Initiator', 'Sentinel']:
    agents_in_role = [ag for ag, r in AGENT_ROLES.items() if r == role]
    role_col = f"{role}_diff"
    
    # Count how many agents of this role each team has
    team_a_count = 0
    team_b_count = 0
    
    for agent in agents_in_role:
        if f"{agent}_t1" in df_train.columns:
            team_a_count += df_train[f"{agent}_t1"]
        if f"{agent}_t2" in df_train.columns:
            team_b_count += df_train[f"{agent}_t2"]
    
    df_train[role_col] = team_a_count - team_b_count
    role_features[role_col] = role_col

print(f"✓ Created {len(role_features)} role-based differential features: {list(role_features.keys())}")

# ============================================================================
# PHASE 3: SYNERGY INTERACTION FEATURES
# ============================================================================

print("\n" + "=" * 70)
print("PHASE 3: SYNERGY INTERACTION FEATURES")
print("=" * 70)

synergy_cols = []

for agent1, agent2 in AGENT_SYNERGIES:
    synergy_name = f"{agent1}_{agent2}_synergy"
    
    # Check if both agents are in the training columns
    agent1_t1_col = f"{agent1}_t1"
    agent2_t1_col = f"{agent2}_t1"
    agent1_t2_col = f"{agent1}_t2"
    agent2_t2_col = f"{agent2}_t2"
    
    if all(col in df_train.columns for col in [agent1_t1_col, agent2_t1_col, agent1_t2_col, agent2_t2_col]):
        # Synergy: +1 if both agents in team A, -1 if both in team B, 0 otherwise
        team_a_synergy = (df_train[agent1_t1_col] * df_train[agent2_t1_col]).astype(int)
        team_b_synergy = (df_train[agent1_t2_col] * df_train[agent2_t2_col]).astype(int)
        df_train[synergy_name] = team_a_synergy - team_b_synergy
        synergy_cols.append(synergy_name)

print(f"✓ Created {len(synergy_cols)} synergy features: {synergy_cols[:5]}...")

# ============================================================================
# PHASE 4: AGENT-MAP WEIGHTING FEATURES
# ============================================================================

print("\n" + "=" * 70)
print("PHASE 4: AGENT-MAP WEIGHTING FEATURES")
print("=" * 70)

# Calculate global agent-map win rates
agent_map_winrates = {}

for agent in AGENT_ROLES.keys():
    for map_name in valid_maps:
        # Find rows where this agent was played on this map
        agent_col_t1 = f"{agent}_t1"
        agent_col_t2 = f"{agent}_t2"
        map_col = f"map_{map_name}"
        
        if agent_col_t1 in df_train.columns and map_col in df_train.columns:
            # Team A played agent
            team_a_mask = (df_train[agent_col_t1] == 1) & (df_train[map_col] == 1)
            if team_a_mask.sum() > 0:
                team_a_wins = (df_train[team_a_mask]['team_a_won_map'] == 1).sum()
                agent_map_winrates[(agent, map_name, 'A')] = team_a_wins / team_a_mask.sum()
        
        if agent_col_t2 in df_train.columns and map_col in df_train.columns:
            # Team B played agent
            team_b_mask = (df_train[agent_col_t2] == 1) & (df_train[map_col] == 1)
            if team_b_mask.sum() > 0:
                team_b_wins = (df_train[team_b_mask]['team_a_won_map'] == 0).sum()
                agent_map_winrates[(agent, map_name, 'B')] = team_b_wins / team_b_mask.sum()

# Create weighted features: convert win rate to differential scale [-1, 1]
for diff_col in diff_cols:
    agent_name = diff_col.replace('_diff', '')
    agent_map_weight_col = f"{agent_name}_map_weight"
    
    # Initialize with zeros
    df_train[agent_map_weight_col] = 0.0
    
    # For each map, calculate weighted difference
    for map_col in map_cols:
        map_name = map_col.replace('map_', '')
        
        # Get win rates for this agent on this map
        team_a_wr = agent_map_winrates.get((agent_name, map_name, 'A'), 0.5)
        team_b_wr = agent_map_winrates.get((agent_name, map_name, 'B'), 0.5)
        
        # Convert to differential: (0, 1) -> (-1, 1)
        team_a_diff = (team_a_wr - 0.5) * 2
        team_b_diff = (team_b_wr - 0.5) * 2
        
        mask = df_train[map_col] == 1
        df_train.loc[mask, agent_map_weight_col] = team_a_diff - team_b_diff

print(f"✓ Created {len(diff_cols)} agent-map weighting features")

# ============================================================================
# BUILD FEATURE SET
# ============================================================================

print("\n" + "=" * 70)
print("BUILDING FEATURE SET")
print("=" * 70)

# Collect all features (excluding one-hot map columns)
all_diff_cols = diff_cols.copy()
all_map_weight_cols = [f"{agent}_map_weight" for agent in AGENT_ROLES.keys() if f"{agent}_diff" in diff_cols]
# IMPORTANT: Do NOT include map_cols in training features
# The model should only learn from differential features, not from the map itself
# Map selection is used to calculate agent_map_weights, but not as a feature
feature_cols = all_diff_cols + list(role_features.keys()) + synergy_cols + all_map_weight_cols

print(f"Feature breakdown:")
print(f"  - Agent differential features: {len(all_diff_cols)}")
print(f"  - Role features: {len(role_features)}")
print(f"  - Synergy features: {len(synergy_cols)}")
print(f"  - Agent-map weight features: {len(all_map_weight_cols)}")
print(f"  - TOTAL: {len(feature_cols)} features")
print(f"  ✓ Dropped one-hot map columns (map_ascent, map_bind, etc.) - only differential features are used for training")

# ============================================================================
# SYMMETRIC TRAINING DATA (UNBIASED)
# ============================================================================

print("\n" + "=" * 70)
print("BUILDING SYMMETRIC TRAINING SET")
print("=" * 70)

df_normal = df_train[feature_cols].copy()
df_normal['target'] = df_train['team_a_won_map']

df_swapped = df_train[feature_cols].copy()
# Swap all differential features (reverse signs)
# Note: All features in feature_cols are now differential, so negate all
for col in feature_cols:
    df_swapped[col] = df_swapped[col] * -1
df_swapped['target'] = 1 - df_train['team_a_won_map']

df_symmetric = pd.concat([df_normal, df_swapped], ignore_index=True)
df_symmetric = df_symmetric.dropna()

X = df_symmetric.drop('target', axis=1)
y = df_symmetric['target']

# Explicitly drop all one-hot encoded map columns (map_ascent, map_bind, etc.)
# Keep only differential features (_diff), synergy features (_synergy), and weight features (_weight)
X = X.drop(columns=[col for col in X.columns if col.startswith('map_') and 'Agent_Map_Weight' not in col], errors='ignore')

print(f"✓ Symmetric training set: {len(X)} samples")

# ============================================================================
# MODEL TRAINING WITH LOGISTIC REGRESSION (fit_intercept=False)
# ============================================================================

print("\n" + "=" * 70)
print("PHASE 1: MODEL TRAINING - LOGISTIC REGRESSION")
print("=" * 70)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# LogisticRegression with fit_intercept=False ensures 50% prob for identical teams
model = LogisticRegression(
    fit_intercept=False,  # Critical: ensures 0 input -> 0.5 probability
    max_iter=1000,
    random_state=42,
    solver='lbfgs'
)
model.fit(X_train, y_train)

train_acc = accuracy_score(y_train, model.predict(X_train))
test_acc = accuracy_score(y_test, model.predict(X_test))
test_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])

print(f"✓ Model trained successfully")
print(f"  - Train Accuracy: {train_acc * 100:.2f}%")
print(f"  - Test Accuracy: {test_acc * 100:.2f}%")
print(f"  - Test AUC-ROC: {test_auc:.4f}")

# Verify symmetry: all-zero input should give ~0.5 probability
zero_input = pd.DataFrame([{col: 0 for col in feature_cols}])
zero_prob = model.predict_proba(zero_input)[0][1]
print(f"  - Symmetry check (0 input -> P(Team1 wins)): {zero_prob:.4f}")

# ============================================================================
# SAVE MODELS & METADATA
# ============================================================================

print("\n" + "=" * 70)
print("SAVING MODELS & METADATA")
print("=" * 70)

joblib.dump(model, 'valorant_lr_model.pkl')
joblib.dump(list(X.columns), 'model_columns.pkl')
joblib.dump(AGENT_ROLES, 'agent_roles.pkl')
joblib.dump(AGENT_SYNERGIES, 'agent_synergies.pkl')
joblib.dump(agent_map_winrates, 'agent_map_winrates.pkl')

print(f"✓ Saved: valorant_lr_model.pkl")
print(f"✓ Saved: model_columns.pkl")
print(f"✓ Saved: agent_roles.pkl")
print(f"✓ Saved: agent_synergies.pkl")
print(f"✓ Saved: agent_map_winrates.pkl")

# ============================================================================
# PHASE 1 FIXED: MAP-AGENT STATISTICS (CORRECTED LOGIC)
# ============================================================================
map_agent_stats = {}
for map_name in valid_maps:
    map_col = f"map_{map_name}"
    if map_col not in df_train.columns:
        continue
        
    map_data = df_train[df_train[map_col] == 1]
    if len(map_data) == 0: 
        continue
        
    agent_win_rates = {}
    for agent in AGENT_ROLES.keys():
        agent_t1 = f"{agent}_t1"
        agent_t2 = f"{agent}_t2"
        
        wins = 0
        plays = 0
        
        # Takım 1'in oynadığı ve kazandığı maçlar
        if agent_t1 in map_data.columns:
            played_t1 = map_data[map_data[agent_t1] == 1]
            plays += len(played_t1)
            wins += played_t1['team_a_won_map'].sum()
            
        # Takım 2'nin oynadığı ve kazandığı maçlar
        if agent_t2 in map_data.columns:
            played_t2 = map_data[map_data[agent_t2] == 1]
            plays += len(played_t2)
            wins += (len(played_t2) - played_t2['team_a_won_map'].sum())
            
        # İstatistiksel geçerlilik için en az 3 maç barajı
        if plays >= 3: 
            agent_win_rates[agent] = round((wins / plays) * 100, 1)
            
    # Haritadaki en başarılı 5 ajanı sırala
    sorted_agents = sorted(agent_win_rates.items(), key=lambda item: item[1], reverse=True)[:5]
    map_agent_stats[map_name] = sorted_agents
# ============================================================================
# FINAL FIX: CLEANING, RE-TRAINING & SAVING
# ============================================================================

print("\n" + "=" * 70)
print("FINAL STEP: CLEANING, RE-TRAINING & SAVING")
print("=" * 70)

# 1. Sütunları son kez temizle ve kesin olarak doğrula
# Harita ismi ile başlayan ancak içerisinde 'weight' geçmeyen (yani one-hot map_ sütunları) tüm sütunları eliyoruz.
final_columns = [
    col for col in X.columns 
    if not (col.startswith('map_') and 'weight' not in col.lower())
]

# 2. X ve eğitim verisini sadece temizlenmiş sütunlarla sınırla
X_final = X[final_columns]
X_train_final = X_train[final_columns]

# 3. Modeli son (temiz) haliyle yeniden eğit
model = LogisticRegression(fit_intercept=False, max_iter=1000, random_state=42)
model.fit(X_train_final, y_train)

# 4. PKL Dosyalarını ZORLA Üzerine Yaz
import joblib
joblib.dump(model, 'valorant_lr_model.pkl')
joblib.dump(list(X_final.columns), 'model_columns.pkl')

# Diğer gerekli dosyaları da güncel tutalım
joblib.dump(AGENT_ROLES, 'agent_roles.pkl')
joblib.dump(AGENT_SYNERGIES, 'agent_synergies.pkl')
joblib.dump(agent_map_winrates, 'agent_map_winrates.pkl')
joblib.dump(map_agent_stats, 'map_agent_stats.pkl')

print(f"✅ Başarılı! Toplam {len(final_columns)} sütun kaydedildi.")
print(f"  - Harita isimli one-hot sütunları (map_ascent vb.) tamamen temizlendi.")
print(f"  - Model yalnızca _diff, _synergy ve _weight sütunları ile eğitildi.")


PHASE 1: DATA LOADING & CLEANING
MATCHES columns: ['match_id', 'event', 'date', 'patch', 'bo_type', 'team_a', 'team_b', 'winner', 'score_a', 'score_b', 'maps_played', 'source']
MAPS columns: ['map_id', 'match_id', 'map_name', 'map_order', 'team_a_score', 'team_b_score', 'attacker_start', 'duration_seconds', 'source']
PLAYERS columns: ['map_id', 'player', 'team', 'agent', 'kills', 'deaths', 'assists', 'acs', 'adr', 'hs_percent', 'kd_ratio', 'source']
✓ Training data rows: 277

PHASE 1: DIFFERENTIAL FEATURES & MODEL ARCHITECTURE
✓ Created 21 individual agent differential features

PHASE 2: ROLE-BASED FEATURES
✓ Created 4 role-based differential features: ['Duelist_diff', 'Controller_diff', 'Initiator_diff', 'Sentinel_diff']

PHASE 3: SYNERGY INTERACTION FEATURES
✓ Created 9 synergy features: ['Jett_Sova_synergy', 'Raze_Fade_synergy', 'Viper_Cypher_synergy', 'Omen_Sage_synergy', 'Brimstone_Killjoy_synergy']...

PHASE 4: AGENT-MAP WEIGHTING FEATURES
✓ Created 21 agent-map weighting feature

NameError: name 'map_agent_stats' is not defined